In [1]:
import glob
import io
import os
import shutil
import sys
import time
import pathlib
import re
from collections import OrderedDict
from datetime import datetime, timedelta
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import polars as pl
import pyautogui
import win32clipboard

from PIL import Image, ImageGrab

from webdriver_manager.chrome import ChromeDriverManager
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import (
    NoAlertPresentException,
    NoSuchElementException,
    TimeoutException,
    StaleElementReferenceException
)

In [2]:
def input_data(data_dir):
    list_files = []
    for path in pathlib.Path(data_dir).glob('**/*.*'):
        if path.suffix.lower() in ['.xlsx', '.csv'] and path.stat().st_size > 0:
            export_time = datetime.fromtimestamp(path.stat().st_mtime)
            try:
                df = pl.read_excel(path) if path.suffix.lower() == '.xlsx' else pl.read_csv(path, infer_schema_length=10000, ignore_errors=True)
                if not df.is_empty():
                    df = df.with_columns([
                        pl.lit(path.stem).alias('sheet_name'),
                        pl.lit(export_time).alias('Export time')
                    ])
                    list_files.append(df)
            except:
                continue
                
    return pl.concat(list_files, how="diagonal_relaxed") if list_files else pl.DataFrame()

In [3]:
first_glob = os.path.expanduser("~").replace("\\", "/")

folder_paths = {
    "req": f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OU.xlsx",
    "current_agents": f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/CAPTURE/current_agent/",
    "input_iex_intervals":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_AGENT_IEX_INTERVALS',

}

In [4]:
def process_ou_hours(file_path):
    try:
        df_raw = pl.read_excel(source=file_path, has_header=False, infer_schema_length=0)
        
        header_vals = df_raw.row(0)
        new_columns = [
            val.strftime("%Y-%m-%d") if isinstance(val, datetime) else str(val).strip() if val is not None else "Unknown" 
            for val in header_vals
        ]
        df_raw.columns = new_columns
        
        df = df_raw.slice(1).with_columns(
            pl.col("Interval").cast(pl.String).str.replace(r"^.*1899-12-31\s+", "").str.slice(0, 8).alias("Interval")
        )
        
        final_df = (df.unpivot(index=["LOB", "Site", "Interval"], variable_name="Date_Str", value_name="Value")
            .with_columns([
                (pl.col("Date_Str").str.slice(0, 10) + " " + pl.col("Interval"))
                .str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S", strict=False)
                .dt.truncate("1m").alias("PST_Datetime"),
                
                pl.col("Value").cast(pl.Float64, strict=False).fill_null(0.0).alias("OU Req by Site"),
                pl.col("LOB").str.strip_chars(), 
                pl.col("Site").str.strip_chars()
            ])
            .with_columns(
                pl.col("PST_Datetime")
                .dt.replace_time_zone("America/Los_Angeles", non_existent="null", ambiguous="earliest")
                .alias("_tz_aware")
            )
            .with_columns([
                pl.col("_tz_aware").dt.convert_time_zone("Asia/Ho_Chi_Minh").dt.replace_time_zone(None).alias("VNT_Datetime"),
                pl.col("_tz_aware").dt.convert_time_zone("Africa/Cairo").dt.replace_time_zone(None).alias("CLT_Datetime"),
                pl.col("_tz_aware").dt.convert_time_zone("Asia/Kolkata").dt.replace_time_zone(None).alias("IST_Datetime"),
                pl.col("OU Req by Site").sum().over(["LOB", "PST_Datetime"]).alias("Total OU Req")
            ])
            .drop("_tz_aware")
        )
        return final_df.select(["LOB", "Site", "PST_Datetime", "VNT_Datetime", "CLT_Datetime", "IST_Datetime", "Total OU Req", "OU Req by Site"])
    except Exception as e:
        print(f"Error processing file: {e}")
        return pl.DataFrame()

req_path = os.path.join(first_glob, r"Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\OU.xlsx")
df_req = process_ou_hours(folder_paths["req"])
df_req

LOB,Site,PST_Datetime,VNT_Datetime,CLT_Datetime,IST_Datetime,Total OU Req,OU Req by Site
str,str,datetime[μs],datetime[μs],datetime[μs],datetime[μs],f64,f64
"""Lodging chat""","""Concentrix (Pune)""",2025-12-29 00:00:00,2025-12-29 15:00:00,2025-12-29 10:00:00,2025-12-29 13:30:00,4.215,1.76
"""Lodging chat""","""Concentrix (Pune)""",2025-12-29 00:30:00,2025-12-29 15:30:00,2025-12-29 10:30:00,2025-12-29 14:00:00,4.215,1.79
"""Lodging chat""","""Concentrix (Pune)""",2025-12-29 01:00:00,2025-12-29 16:00:00,2025-12-29 11:00:00,2025-12-29 14:30:00,4.215,2.165
"""Lodging chat""","""Concentrix (Pune)""",2025-12-29 01:30:00,2025-12-29 16:30:00,2025-12-29 11:30:00,2025-12-29 15:00:00,4.215,1.96
"""Lodging chat""","""Concentrix (Pune)""",2025-12-29 02:00:00,2025-12-29 17:00:00,2025-12-29 12:00:00,2025-12-29 15:30:00,4.215,2.29
…,…,…,…,…,…,…,…
"""Non-Lodging chat""","""Concentrix (Kolkata)""",2026-05-10 21:30:00,2026-05-11 11:30:00,2026-05-11 07:30:00,2026-05-11 10:00:00,0.0,0.0
"""Non-Lodging chat""","""Concentrix (Kolkata)""",2026-05-10 22:00:00,2026-05-11 12:00:00,2026-05-11 08:00:00,2026-05-11 10:30:00,0.0,0.0
"""Non-Lodging chat""","""Concentrix (Kolkata)""",2026-05-10 22:30:00,2026-05-11 12:30:00,2026-05-11 08:30:00,2026-05-11 11:00:00,0.0,0.0


In [5]:
IEX = input_data(folder_paths["input_iex_intervals"])
IEX

,Month,Week_Monday,Date_Converted,Employee Name,OracleID,IEX ID,Email Id,First Shift,Scheduled Activity,VNT_Intervals,PST_Intervals,VNT_Interval_Range,PST_Interval_Range,Datetime_Start_Time,Datetime_End_Time,Duration,Work Category,sheet_name,Export time
i64,str,str,str,str,i64,i64,str,str,str,str,str,str,str,str,str,f64,str,str,datetime[μs]
0,"""Dec-25""","""2025-12-01""","""2025-12-01""","""HOANG THI PHUONG HOA""",102484482,3000182,"""thiphuonghoa.hoang@concentrix.…","""2100-0600""","""Open Time""","""2025-12-01 21:00:00""","""2025-12-01 06:00:00""","""21:00-21:30""","""06:00-06:30""","""2025-12-01 21:00:00""","""2025-12-01 21:30:00""",0.5,"""Productive""","""2025-12-01""",2026-04-07 10:30:58
1,"""Dec-25""","""2025-12-01""","""2025-12-01""","""HOANG THI PHUONG HOA""",102484482,3000182,"""thiphuonghoa.hoang@concentrix.…","""2100-0600""","""Open Time""","""2025-12-01 21:30:00""","""2025-12-01 06:30:00""","""21:30-22:00""","""06:30-07:00""","""2025-12-01 21:30:00""","""2025-12-01 22:00:00""",0.5,"""Productive""","""2025-12-01""",2026-04-07 10:30:58
2,"""Dec-25""","""2025-12-01""","""2025-12-01""","""HOANG THI PHUONG HOA""",102484482,3000182,"""thiphuonghoa.hoang@concentrix.…","""2100-0600""","""Open Time""","""2025-12-01 22:00:00""","""2025-12-01 07:00:00""","""22:00-22:30""","""07:00-07:30""","""2025-12-01 22:00:00""","""2025-12-01 22:15:00""",0.25,"""Productive""","""2025-12-01""",2026-04-07 10:30:58
3,"""Dec-25""","""2025-12-01""","""2025-12-01""","""HOANG THI PHUONG HOA""",102484482,3000182,"""thiphuonghoa.hoang@concentrix.…","""2100-0600""","""Break_1""","""2025-12-01 22:00:00""","""2025-12-01 07:00:00""","""22:00-22:30""","""07:00-07:30""","""2025-12-01 22:15:00""","""2025-12-01 22:30:00""",0.25,"""Unproductive""","""2025-12-01""",2026-04-07 10:30:58
4,"""Dec-25""","""2025-12-01""","""2025-12-01""","""HOANG THI PHUONG HOA""",102484482,3000182,"""thiphuonghoa.hoang@concentrix.…","""2100-0600""","""Open Time""","""2025-12-01 22:30:00""","""2025-12-01 07:30:00""","""22:30-23:00""","""07:30-08:00""","""2025-12-01 22:30:00""","""2025-12-01 23:00:00""",0.5,"""Productive""","""2025-12-01""",2026-04-07 10:30:58
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
37193,"""May-26""","""2026-05-04""","""2026-05-10""","""DUONG THI THUY DUONG""",103305150,3112809,"""thithuyduong.duong@concentrix.…","""2000-0500""","""Open Time""","""2026-05-11 03:00:00""","""2026-05-10 13:00:00""","""03:00-03:30""","""13:00-13:30""","""2026-05-11 03:00:00""","""2026-05-11 03:15:00""",0.25,"""Productive""","""2026-05-04""",2026-05-05 14:10:19
37194,"""May-26""","""2026-05-04""","""2026-05-10""","""DUONG THI THUY DUONG""",103305150,3112809,"""thithuyduong.duong@concentrix.…","""2000-0500""","""Break_2""","""2026-05-11 03:00:00""","""2026-05-10 13:00:00""","""03:00-03:30""","""13:00-13:30""","""2026-05-11 03:15:00""","""2026-05-11 03:30:00""",0.25,"""Unproductive""","""2026-05-04""",2026-05-05 14:10:19
37195,"""May-26""","""2026-05-04""","""2026-05-10""","""DUONG THI THUY DUONG""",103305150,3112809,"""thithuyduong.duong@concentrix.…","""2000-0500""","""Open Time""","""2026-05-11 03:30:00""","""2026-05-10 13:30:00""","""03:30-04:00""","""13:30-14:00""","""2026-05-11 03:30:00""","""2026-05-11 04:00:00""",0.5,"""Productive""","""2026-05-04""",2026-05-05 14:10:19


In [6]:
outage_db = input_data(folder_paths['current_agents'])

latest16 = outage_db.select("Export time").unique().sort("Export time", descending=True).limit(16)

outage_db = outage_db.filter(pl.col("Export time").is_in(latest16.get_column("Export time")))

outage_db = outage_db.select([
    'Business Location', 'Export time', 'Agent Name', 'Agent Manager', 
    'Connect State', 'Assigned Workitem Count', 'Duration', 'Queue Group / Routing Profile'
])

outage_db = outage_db.with_columns(
    pl.col("Export time").cast(pl.Datetime, strict=False).dt.truncate("30m").alias("VN_Datetime")
)

outage_db = outage_db.with_columns([
    (pl.col("VN_Datetime") - pl.duration(hours=14)).alias("PST_Datetime"),
    pl.col("VN_Datetime").dt.date().alias("VN_Date"),
    pl.col("VN_Datetime").dt.strftime("%H:%M").alias("VN_Intervals")
])

outage_db = outage_db.with_columns([
    pl.col("PST_Datetime").dt.date().alias("PST_Date"),
    pl.col("PST_Datetime").dt.strftime("%H:%M").alias("PST_Intervals"),
    pl.col('Duration').str.strptime(pl.Time, format='%H:%M:%S', strict=False)
])

outage_db = outage_db.with_columns(
    pl.when(pl.col("Queue Group / Routing Profile").is_in(["Chat_OD_EN_Car_Activity", "Chat_OD_EN_Lodging", "Chat - Global English Lodging Nesting", "Chat_Lodging English w Car"])).then(pl.lit("Lodging Chat"))
    .when(pl.col("Queue Group / Routing Profile").is_in(["Chat - Global English Non- Lodging Nesting", "Chat_OD_EN_Dual_GDS"])).then(pl.lit("Non Lodging Chat"))
    .otherwise(None).alias("LOB")
)

all_site_outage = outage_db.filter(
    pl.col("LOB").is_not_null() & (pl.col("LOB").str.strip_chars() != "")
)

all_site_outage = all_site_outage.with_columns(
    (pl.col("Duration").dt.hour() * 3600 + pl.col("Duration").dt.minute() * 60 + pl.col("Duration").dt.second()).cast(pl.Int32).alias("duration_seconds")
)

active_cond = pl.col("Connect State").is_in(["AVAILABLECHAT", "OUTBOUNDCALL"]) | (pl.col("Assigned Workitem Count").cast(pl.Int32, strict=False).fill_null(0) > 0)
inactive_cond = ~active_cond

aggs_active = [
    pl.col("Agent Name").filter(pl.col("Business Location").str.contains(loc) & active_cond).n_unique().alias(f"{prefix} Heads Active")
    for loc, prefix in [("Pune", "PUN"), ("Ho Chi Minh", "HCM"), ("Kolkata", "KOL"), ("Cairo", "CAI")]
]

df_active = all_site_outage.group_by(["LOB", "PST_Datetime"]).agg(aggs_active).fill_null(0).sort(["LOB", "PST_Datetime"])

aggs_inactive = [
    pl.col("Agent Name").filter(pl.col("Business Location").str.contains(loc) & inactive_cond).n_unique().alias(f"{prefix} Heads Inactive")
    for loc, prefix in [("Pune", "PUN"), ("Ho Chi Minh", "HCM"), ("Kolkata", "KOL"), ("Cairo", "CAI")]
]

df_inactive = all_site_outage.group_by(["LOB", "PST_Datetime"]).agg(aggs_inactive).fill_null(0).sort(["LOB", "PST_Datetime"])

df_inactive_details = all_site_outage.filter(inactive_cond).select([
    "Agent Name", "LOB", "Business Location", "Connect State", "Duration", "PST_Datetime"
])

df_inactive_details = df_inactive_details.rename({"PST_Datetime": "PST_Interval"})

df_inactive_details = df_inactive_details.with_columns(
    pl.col("PST_Interval").dt.replace_time_zone("America/Los_Angeles", non_existent="null", ambiguous="earliest").alias("_tz_aware")
).with_columns([
    pl.col("PST_Interval").dt.date().alias("PST_Date"),
    pl.col("_tz_aware").dt.convert_time_zone("Asia/Kolkata").dt.replace_time_zone(None).dt.strftime("%H:%M").alias("IST"),
    pl.col("_tz_aware").dt.convert_time_zone("Africa/Cairo").dt.replace_time_zone(None).dt.strftime("%H:%M").alias("CLT"),
    pl.col("_tz_aware").dt.convert_time_zone("Asia/Ho_Chi_Minh").dt.replace_time_zone(None).dt.strftime("%H:%M").alias("VNT"),
    pl.col("PST_Interval").dt.strftime("%H:%M").alias("PST")
]).drop("_tz_aware", "PST_Interval")

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_19960\3996463841.py:5: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  outage_db = outage_db.filter(pl.col("Export time").is_in(latest16.get_column("Export time")))


In [7]:
df_latest_inactive = df_inactive_details.filter((pl.col("PST") == pl.col("PST").max())  ).sort(["LOB", "Business Location"])

FILTER_LATEST_PST = True

latest_pst_val = df_latest_inactive.select(pl.col("PST").first()).item()

if FILTER_LATEST_PST and latest_pst_val is not None:
    df_active = df_active.filter(pl.col("PST_Datetime").dt.strftime("%H:%M") == latest_pst_val)
    df_inactive = df_inactive.filter(pl.col("PST_Datetime").dt.strftime("%H:%M") == latest_pst_val)

df_lodging = df_latest_inactive.filter(pl.col("LOB") == "Lodging Chat")
df_non_lodging = df_latest_inactive.filter(pl.col("LOB") == "Non Lodging Chat")

with pl.Config(tbl_rows=-1):
    display(df_lodging)
    display(df_non_lodging)
    display(df_active)
    display(df_inactive)

Agent Name,LOB,Business Location,Connect State,Duration,PST_Date,IST,CLT,VNT,PST
str,str,str,str,time,date,str,str,str,str
"""Chau, Thien Kim""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""LOGIN""",00:58:44,2026-05-04,"""08:30""","""06:00""","""10:00""","""20:00"""
"""Huynh, Quang Anh""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""ENDOFSHIFT""",04:14:08,2026-05-04,"""08:30""","""06:00""","""10:00""","""20:00"""
"""Le, MINH QUAN""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""TRAINING""",01:13:14,2026-05-04,"""08:30""","""06:00""","""10:00""","""20:00"""
"""Ngo, Ha Trang""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""LUNCH""",00:04:53,2026-05-04,"""08:30""","""06:00""","""10:00""","""20:00"""
"""Ngo, Tan Phat""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""LUNCH""",00:03:14,2026-05-04,"""08:30""","""06:00""","""10:00""","""20:00"""
"""Nguyen, Thi Anh Thu""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""LOGIN""",08:46:10,2026-05-04,"""08:30""","""06:00""","""10:00""","""20:00"""
"""Nguyen, Thi Thien An""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""TRAINING""",01:11:12,2026-05-04,"""08:30""","""06:00""","""10:00""","""20:00"""
"""Nguyen, Tran Anh Thu""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""LUNCH""",00:13:47,2026-05-04,"""08:30""","""06:00""","""10:00""","""20:00"""
"""Pham, Thi Tuyet Nhi""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""LOGIN""",17:48:51,2026-05-04,"""08:30""","""06:00""","""10:00""","""20:00"""


Agent Name,LOB,Business Location,Connect State,Duration,PST_Date,IST,CLT,VNT,PST
str,str,str,str,time,date,str,str,str,str
"""Reda abdelgawad elmasry, Ahmed""","""Non Lodging Chat""","""Concentrix (Cairo)""","""ENDOFSHIFT""",00:09:20,2026-05-04,"""08:30""","""06:00""","""10:00""","""20:00"""


LOB,PST_Datetime,PUN Heads Active,HCM Heads Active,KOL Heads Active,CAI Heads Active
str,datetime[μs],u32,u32,u32,u32
"""Lodging Chat""",2026-05-04 20:00:00,1,18,2,0
"""Non Lodging Chat""",2026-05-04 20:00:00,0,0,5,4


LOB,PST_Datetime,PUN Heads Inactive,HCM Heads Inactive,KOL Heads Inactive,CAI Heads Inactive
str,datetime[μs],u32,u32,u32,u32
"""Lodging Chat""",2026-05-04 20:00:00,1,11,4,0
"""Non Lodging Chat""",2026-05-04 20:00:00,0,0,0,1


In [8]:
from IPython.display import display, HTML

# Tạo cấu trúc HTML để chia 2 cột hiển thị
html_str = f"""
<div style="display:flex; justify-content: space-around; gap: 20px;">
    <div style="flex: 1;">
        <h3 style="text-align: center; color: #2c3e50;">🛌 Lodging Chat (Latest PST)</h3>
        {df_lodging._repr_html_()}
    </div>
    <div style="flex: 1;">
        <h3 style="text-align: center; color: #2c3e50;">💬 Non Lodging Chat (Latest PST)</h3>
        {df_non_lodging._repr_html_()}
    </div>
</div>
<hr style="margin: 30px 0;">
<div style="display:flex; justify-content: space-around; gap: 20px;">
    <div style="flex: 1;">
        <h3 style="text-align: center; color: #27ae60;">🟢 Active Summary</h3>
        {df_active._repr_html_()}
    </div>
    <div style="flex: 1;">
        <h3 style="text-align: center; color: #e74c3c;">🔴 Inactive Summary</h3>
        {df_inactive._repr_html_()}
    </div>
</div>
"""

# Hiển thị ra màn hình
display(HTML(html_str))

Agent Name,LOB,Business Location,Connect State,Duration,PST_Date,IST,CLT,VNT,PST
str,str,str,str,time,date,str,str,str,str
"""Chau, Thien Kim""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""LOGIN""",00:58:44,2026-05-04,"""08:30""","""06:00""","""10:00""","""20:00"""
"""Huynh, Quang Anh""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""ENDOFSHIFT""",04:14:08,2026-05-04,"""08:30""","""06:00""","""10:00""","""20:00"""
"""Le, MINH QUAN""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""TRAINING""",01:13:14,2026-05-04,"""08:30""","""06:00""","""10:00""","""20:00"""
"""Ngo, Ha Trang""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""LUNCH""",00:04:53,2026-05-04,"""08:30""","""06:00""","""10:00""","""20:00"""
"""Ngo, Tan Phat""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""LUNCH""",00:03:14,2026-05-04,"""08:30""","""06:00""","""10:00""","""20:00"""
…,…,…,…,…,…,…,…,…,…
"""Faizan, S""","""Lodging Chat""","""Concentrix (Kolkata)""","""RESTROOM""",00:19:24,2026-05-04,"""08:30""","""06:00""","""10:00""","""20:00"""
"""PAUL, ANASUA""","""Lodging Chat""","""Concentrix (Kolkata)""","""LUNCH""",00:08:18,2026-05-04,"""08:30""","""06:00""","""10:00""","""20:00"""
"""Shaw, SIDDHARTHA""","""Lodging Chat""","""Concentrix (Kolkata)""","""ENDOFSHIFT""",07:26:56,2026-05-04,"""08:30""","""06:00""","""10:00""","""20:00"""


In [9]:
latest_export = outage_db.select(pl.col("Export time").cast(pl.Datetime, strict=False).max()).item()
last_refresh_txt = latest_export.strftime("%b %d, %Y %H:%M") if latest_export is not None else "N/A"
print("Last refresh (VN):", last_refresh_txt)

Last refresh (VN): May 05, 2026 10:14
